## 🎯 Learning Objectives
* Design and implement a robust evaluation harness for AI agents.
* Define clear test cases and expected outcomes for agent behavior.
* Calculate and interpret key performance metrics for agent evaluation.
* Generate comprehensive reports to summarize agent performance.
* Understand the importance of modularity and extensibility in evaluation systems.


## Exercise: Write an Evaluation Harness for Your Agent

### Task Description

In this exercise, you will design and implement a comprehensive evaluation harness for an AI agent. An evaluation harness is a critical component for systematically testing, benchmarking, and improving agent performance. It allows you to define a set of test cases, execute your agent against them, and measure its effectiveness based on predefined criteria.

Your harness should be capable of assessing various aspects of an agent's behavior, including its ability to produce correct final answers, make appropriate tool calls, and operate efficiently.

### Requirements

1.  **Test Case Definition**: The harness must accept a collection of test cases. Each test case should include:
    *   An `input` prompt for the agent.
    *   An `expected_output` (the desired final answer).
    *   `expected_tool_calls` (an optional list of dictionaries representing the sequence of tool calls the agent is expected to make, including tool name and arguments).
    *   `expected_success` (a boolean indicating if the agent is expected to succeed for this case).

2.  **Agent Integration**: The harness should be able to run a given agent instance against each test case. Assume the agent has a `run(input: str) -> AgentResponse` method, where `AgentResponse` is a structured object containing the agent's final answer, a log of its internal steps (including tool calls), and any other relevant metadata.

3.  **Metric Calculation**: Implement the following evaluation metrics:
    *   **Overall Success Rate**: The percentage of test cases where the agent's `final_answer` matches the `expected_output`.
    *   **Tool Usage Accuracy**: The percentage of test cases where the agent's sequence of `tool_calls_made` exactly matches `expected_tool_calls` (if provided for the test case).
    *   **Average Latency**: The average time (in seconds) the agent takes to process a test case.
    *   **Robustness Score**: A composite score that considers both final answer correctness and tool usage accuracy. You can define your own weighting.

4.  **Detailed Logging**: For each test case, the harness should log:
    *   The input prompt.
    *   The agent's actual final answer.
    *   The expected final answer.
    *   Whether the final answer was correct.
    *   The agent's actual tool calls.
    *   The expected tool calls.
    *   Whether the tool calls were correct.
    *   The execution time.
    *   Any internal steps or thoughts logged by the agent (if available).

5.  **Report Generation**: The harness must generate a summary report, ideally using a `pandas.DataFrame`, that includes:
    *   Overall metrics (Success Rate, Tool Usage Accuracy, Average Latency, Robustness Score).
    *   A breakdown of results for each individual test case.

6.  **Modularity**: Design your harness with modularity in mind, allowing for easy addition of new metrics or different types of agents in the future.

### Evaluation Criteria

*   **Correctness**: Does the harness accurately calculate all specified metrics and correctly compare agent outputs/tool calls against expected values?
*   **Completeness**: Are all requirements met, including detailed logging and report generation?
*   **Code Quality**: Is the code clean, well-structured, readable, and appropriately commented? Does it follow modern Python best practices (e.g., type hints, Pydantic for data models)?
*   **Robustness**: Does the harness handle cases where `expected_tool_calls` might be `None` or empty?
*   **Report Clarity**: Is the generated report easy to understand and informative?


In [ ]:
import time
import random
from typing import List, Dict, Any, Optional, Callable
from pydantic import BaseModel, Field
import pandas as pd

# --- Setup Code: Data Models and Mock Agent --- 

# 2026 Ready: Using Pydantic v2 for robust data modeling

class ToolCall(BaseModel):
    tool_name: str
    args: Dict[str, Any]

class AgentStep(BaseModel):
    step_type: str # e.g., 'thought', 'tool_call', 'observation'
    content: Any
    timestamp: float = Field(default_factory=time.time)

class AgentResponse(BaseModel):
    final_answer: str
    tool_calls_made: List[ToolCall] = Field(default_factory=list)
    internal_steps_log: List[AgentStep] = Field(default_factory=list)
    execution_time: float = 0.0 # To be filled by the agent or harness

class TestCase(BaseModel):
    id: str
    input: str
    expected_output: str
    expected_tool_calls: Optional[List[ToolCall]] = None
    expected_success: bool = True

class MockAgent:
    """A mock agent for testing the evaluation harness."""
    def __init__(self, name: str = "MockAgent", latency_range: tuple = (0.1, 0.5)):
        self.name = name
        self.latency_range = latency_range

    def run(self, input: str) -> AgentResponse:
        start_time = time.time()
        
        # Simulate agent's internal thought process
        steps = [AgentStep(step_type="thought", content=f"Thinking about input: '{input}'")]

        final_answer = ""
        tool_calls = []

        # Simple logic to simulate different agent behaviors based on input
        if "search for weather" in input.lower():
            steps.append(AgentStep(step_type="tool_call", content=ToolCall(tool_name="weather_api", args={"location": "London"})))
            tool_calls.append(ToolCall(tool_name="weather_api", args={"location": "London"}))
            steps.append(AgentStep(step_type="observation", content="Weather in London: Sunny, 20C"))
            final_answer = "The weather in London is sunny with 20 degrees Celsius."
        elif "calculate 10 + 5" in input.lower():
            steps.append(AgentStep(step_type="tool_call", content=ToolCall(tool_name="calculator", args={"expression": "10 + 5"})))
            tool_calls.append(ToolCall(tool_name="calculator", args={"expression": "10 + 5"}))
            steps.append(AgentStep(step_type="observation", content="Result: 15"))
            final_answer = "The result of 10 + 5 is 15."
        elif "tell me a joke" in input.lower():
            final_answer = "Why don't scientists trust atoms? Because they make up everything!"
        elif "fail this task" in input.lower():
            final_answer = "I am unable to complete this task due to an internal error."
            steps.append(AgentStep(step_type="error", content="Simulated internal error."))
        else:
            final_answer = f"I received your request: '{input}'. I'm still learning!"
        
        # Simulate latency
        time.sleep(random.uniform(*self.latency_range))
        
        end_time = time.time()
        execution_time = end_time - start_time

        return AgentResponse(
            final_answer=final_answer,
            tool_calls_made=tool_calls,
            internal_steps_log=steps,
            execution_time=execution_time
        )

# --- Mock Dataset of Test Cases ---

mock_test_cases: List[TestCase] = [
    TestCase(
        id="TC001",
        input="What is the current weather in London?",
        expected_output="The weather in London is sunny with 20 degrees Celsius.",
        expected_tool_calls=[ToolCall(tool_name="weather_api", args={"location": "London"})]
    ),
    TestCase(
        id="TC002",
        input="Compute the sum of 10 and 5.",
        expected_output="The result of 10 + 5 is 15.",
        expected_tool_calls=[ToolCall(tool_name="calculator", args={"expression": "10 + 5"})]
    ),
    TestCase(
        id="TC003",
        input="Tell me a funny joke.",
        expected_output="Why don't scientists trust atoms? Because they make up everything!",
        expected_tool_calls=[] # No tools expected
    ),
    TestCase(
        id="TC004",
        input="What is the capital of France?",
        expected_output="I received your request: 'What is the capital of France?'. I'm still learning!",
        expected_tool_calls=[],
        expected_success=False # Agent is expected to give a generic answer here
    ),
    TestCase(
        id="TC005",
        input="Please fail this task intentionally.",
        expected_output="I am unable to complete this task due to an internal error.",
        expected_tool_calls=[],
        expected_success=False # Agent is expected to fail
    )
]

print("Setup complete. MockAgent and test cases are ready.")


### Your Turn: Implement the Agent Evaluation Harness

Now, it's your turn to implement the `AgentEvaluationHarness` class. Follow the requirements outlined in the task description. Your implementation should include methods to:

1.  Initialize the harness with a list of `TestCase` objects.
2.  Run the evaluation against a given agent instance.
3.  Collect detailed results for each test case.
4.  Calculate the required aggregate metrics.
5.  Generate a comprehensive report.

Think about how to structure your class to be extensible and easy to use. Pay close attention to how you compare `expected_tool_calls` with `actual_tool_calls` – remember that the order and exact arguments matter for tool usage accuracy.

```python
# Start your implementation here

class AgentEvaluationHarness:
    def __init__(self, test_cases: List[TestCase]):
        self.test_cases = test_cases
        self.results: List[Dict[str, Any]] = []

    def _compare_tool_calls(self, expected: Optional[List[ToolCall]], actual: List[ToolCall]) -> bool:
        # Implement logic to compare tool calls. Consider order and arguments.
        # Return True if they match, False otherwise.
        pass

    def run_evaluation(self, agent: Any):
        # Iterate through test cases, run the agent, collect results.
        pass

    def generate_report(self) -> pd.DataFrame:
        # Calculate aggregate metrics and return a pandas DataFrame report.
        pass

# Example usage (after you implement the class):
# harness = AgentEvaluationHarness(mock_test_cases)
# mock_agent = MockAgent()
# harness.run_evaluation(mock_agent)
# report = harness.generate_report()
# print(report)
```


In [ ]:
import time
import random
from typing import List, Dict, Any, Optional, Callable
from pydantic import BaseModel, Field
import pandas as pd

# Re-define data models for clarity in the solution, though they are already defined above.
# In a real notebook, you'd typically define them once.
class ToolCall(BaseModel):
    tool_name: str
    args: Dict[str, Any]

class AgentStep(BaseModel):
    step_type: str
    content: Any
    timestamp: float = Field(default_factory=time.time)

class AgentResponse(BaseModel):
    final_answer: str
    tool_calls_made: List[ToolCall] = Field(default_factory=list)
    internal_steps_log: List[AgentStep] = Field(default_factory=list)
    execution_time: float = 0.0

class TestCase(BaseModel):
    id: str
    input: str
    expected_output: str
    expected_tool_calls: Optional[List[ToolCall]] = None
    expected_success: bool = True

# MockAgent (as defined in setup, included here for completeness of the solution block)
class MockAgent:
    def __init__(self, name: str = "MockAgent", latency_range: tuple = (0.1, 0.5)):
        self.name = name
        self.latency_range = latency_range

    def run(self, input: str) -> AgentResponse:
        start_time = time.time()
        steps = [AgentStep(step_type="thought", content=f"Thinking about input: '{input}'")]
        final_answer = ""
        tool_calls = []

        if "search for weather" in input.lower():
            steps.append(AgentStep(step_type="tool_call", content=ToolCall(tool_name="weather_api", args={"location": "London"})))
            tool_calls.append(ToolCall(tool_name="weather_api", args={"location": "London"}))
            steps.append(AgentStep(step_type="observation", content="Weather in London: Sunny, 20C"))
            final_answer = "The weather in London is sunny with 20 degrees Celsius."
        elif "calculate 10 + 5" in input.lower():
            steps.append(AgentStep(step_type="tool_call", content=ToolCall(tool_name="calculator", args={"expression": "10 + 5"})))
            tool_calls.append(ToolCall(tool_name="calculator", args={"expression": "10 + 5"}))
            steps.append(AgentStep(step_type="observation", content="Result: 15"))
            final_answer = "The result of 10 + 5 is 15."
        elif "tell me a joke" in input.lower():
            final_answer = "Why don't scientists trust atoms? Because they make up everything!"
        elif "fail this task" in input.lower():
            final_answer = "I am unable to complete this task due to an internal error."
            steps.append(AgentStep(step_type="error", content="Simulated internal error."))
        else:
            final_answer = f"I received your request: '{input}'. I'm still learning!"
        
        time.sleep(random.uniform(*self.latency_range))
        end_time = time.time()
        execution_time = end_time - start_time

        return AgentResponse(
            final_answer=final_answer,
            tool_calls_made=tool_calls,
            internal_steps_log=steps,
            execution_time=execution_time
        )

# Mock test cases (as defined in setup, included here for completeness of the solution block)
mock_test_cases: List[TestCase] = [
    TestCase(
        id="TC001",
        input="What is the current weather in London?",
        expected_output="The weather in London is sunny with 20 degrees Celsius.",
        expected_tool_calls=[ToolCall(tool_name="weather_api", args={"location": "London"})]
    ),
    TestCase(
        id="TC002",
        input="Compute the sum of 10 and 5.",
        expected_output="The result of 10 + 5 is 15.",
        expected_tool_calls=[ToolCall(tool_name="calculator", args={"expression": "10 + 5"})]
    ),
    TestCase(
        id="TC003",
        input="Tell me a funny joke.",
        expected_output="Why don't scientists trust atoms? Because they make up everything!",
        expected_tool_calls=[]
    ),
    TestCase(
        id="TC004",
        input="What is the capital of France?",
        expected_output="I received your request: 'What is the capital of France?'. I'm still learning!",
        expected_tool_calls=[],
        expected_success=False
    ),
    TestCase(
        id="TC005",
        input="Please fail this task intentionally.",
        expected_output="I am unable to complete this task due to an internal error.",
        expected_tool_calls=[],
        expected_success=False
    )
]


class AgentEvaluationHarness:
    """A robust evaluation harness for AI agents."""

    def __init__(self, test_cases: List[TestCase]):
        self.test_cases = test_cases
        self.results: List[Dict[str, Any]] = []

    def _compare_tool_calls(self, expected: Optional[List[ToolCall]], actual: List[ToolCall]) -> bool:
        """Compares two lists of ToolCall objects for exact match (order and arguments)."""
        if not expected and not actual: # Both empty or None, considered a match
            return True
        if not expected or not actual: # One is empty/None, the other is not
            return False
        if len(expected) != len(actual):
            return False
        
        # Compare each tool call in sequence
        for i in range(len(expected)):
            # Pydantic models can be compared directly for equality if their fields match
            if expected[i] != actual[i]:
                return False
        return True

    def run_evaluation(self, agent: Any):
        """Runs the given agent against all test cases and stores detailed results."""
        self.results = [] # Reset results for a new evaluation run

        print(f"\n--- Starting Evaluation for Agent: {getattr(agent, 'name', 'Unnamed Agent')} ---")
        for i, test_case in enumerate(self.test_cases):
            print(f"Running Test Case {test_case.id} ({i+1}/{len(self.test_cases)}): '{test_case.input[:50]}...'", end=" ")
            
            try:
                agent_response = agent.run(test_case.input)
                
                # Evaluate final answer correctness
                final_answer_correct = (agent_response.final_answer == test_case.expected_output)
                
                # Evaluate tool call correctness
                tool_calls_correct = self._compare_tool_calls(
                    test_case.expected_tool_calls,
                    agent_response.tool_calls_made
                )

                # Determine overall success for this test case
                # An agent is 'successful' if its final answer is correct AND its tool calls (if expected) are correct.
                # Also, we check against `test_case.expected_success` to see if the agent performed as expected.
                overall_case_success = final_answer_correct and tool_calls_correct
                
                # The harness itself needs to check if the agent's actual outcome matches the *expected* outcome for the test case.
                # For example, if expected_success is False, and the agent fails, that's a 'correct' outcome for the test case.
                harness_outcome_matches_expected = (overall_case_success == test_case.expected_success)

                self.results.append({
                    "test_case_id": test_case.id,
                    "input": test_case.input,
                    "expected_output": test_case.expected_output,
                    "actual_output": agent_response.final_answer,
                    "final_answer_correct": final_answer_correct,
                    "expected_tool_calls": [tc.model_dump() for tc in test_case.expected_tool_calls] if test_case.expected_tool_calls else [],
                    "actual_tool_calls": [tc.model_dump() for tc in agent_response.tool_calls_made],
                    "tool_calls_correct": tool_calls_correct,
                    "execution_time": agent_response.execution_time,
                    "internal_steps_log": [step.model_dump() for step in agent_response.internal_steps_log],
                    "expected_overall_success": test_case.expected_success,
                    "actual_overall_success": overall_case_success,
                    "harness_outcome_matches_expected": harness_outcome_matches_expected # Did the agent behave as expected by the test case?
                })
                print(f"-> {'PASS' if harness_outcome_matches_expected else 'FAIL'}")

            except Exception as e:
                # Handle unexpected errors during agent execution
                print(f"-> ERROR: {e}")
                self.results.append({
                    "test_case_id": test_case.id,
                    "input": test_case.input,
                    "expected_output": test_case.expected_output,
                    "actual_output": f"ERROR: {e}",
                    "final_answer_correct": False,
                    "expected_tool_calls": [tc.model_dump() for tc in test_case.expected_tool_calls] if test_case.expected_tool_calls else [],
                    "actual_tool_calls": [],
                    "tool_calls_correct": False,
                    "execution_time": 0.0,
                    "internal_steps_log": [AgentStep(step_type="error", content=str(e)).model_dump()],
                    "expected_overall_success": test_case.expected_success,
                    "actual_overall_success": False,
                    "harness_outcome_matches_expected": (False == test_case.expected_success) # If agent errors, it's a failure
                })
        print("--- Evaluation Complete ---")

    def generate_report(self) -> pd.DataFrame:
        """Generates a comprehensive pandas DataFrame report of the evaluation results."""
        if not self.results:
            print("No evaluation results found. Run `run_evaluation` first.")
            return pd.DataFrame()

        df = pd.DataFrame(self.results)

        # Calculate aggregate metrics
        total_cases = len(df)
        
        # Overall Success Rate: How often the agent's actual outcome matched the test case's expected outcome
        overall_success_rate = df['harness_outcome_matches_expected'].mean() * 100 if total_cases > 0 else 0
        
        # Agent's Final Answer Correctness Rate (ignoring expected_success for the test case itself)
        agent_final_answer_correctness_rate = df['final_answer_correct'].mean() * 100 if total_cases > 0 else 0

        # Tool Usage Accuracy: Only consider cases where tool calls were expected
        tool_cases = df[df['expected_tool_calls'].apply(lambda x: len(x) > 0)]
        tool_usage_accuracy = tool_cases['tool_calls_correct'].mean() * 100 if len(tool_cases) > 0 else 100.0 # If no tool cases, 100% by default

        average_latency = df['execution_time'].mean() if total_cases > 0 else 0.0

        # Robustness Score: A composite score (e.g., 70% final answer correctness, 30% tool usage accuracy)
        # This is an example; weighting can be adjusted based on specific needs.
        robustness_score = (0.7 * agent_final_answer_correctness_rate) + (0.3 * tool_usage_accuracy)

        # Prepare summary report
        summary_data = {
            "Metric": [
                "Total Test Cases",
                "Overall Evaluation Success Rate (Agent behaved as expected)",
                "Agent Final Answer Correctness Rate",
                "Tool Usage Accuracy (where tools expected)",
                "Average Latency (s)",
                "Robustness Score"
            ],
            "Value": [
                total_cases,
                f"{overall_success_rate:.2f}%",
                f"{agent_final_answer_correctness_rate:.2f}%",
                f"{tool_usage_accuracy:.2f}%",
                f"{average_latency:.3f}",
                f"{robustness_score:.2f}"
            ]
        }
        summary_df = pd.DataFrame(summary_data)

        print("\n--- Evaluation Summary ---")
        print(summary_df.to_string(index=False))
        print("\n--- Detailed Results Per Test Case ---")
        
        # Select and reorder columns for detailed report for better readability
        detailed_df = df[[
            "test_case_id",
            "input",
            "expected_output",
            "actual_output",
            "final_answer_correct",
            "expected_tool_calls",
            "actual_tool_calls",
            "tool_calls_correct",
            "execution_time",
            "expected_overall_success",
            "actual_overall_success",
            "harness_outcome_matches_expected"
        ]]
        # Convert list of dicts to string for better display in pandas
        detailed_df['expected_tool_calls'] = detailed_df['expected_tool_calls'].apply(lambda x: str(x))
        detailed_df['actual_tool_calls'] = detailed_df['actual_tool_calls'].apply(lambda x: str(x))

        print(detailed_df.to_string(index=False))

        return df # Return the raw DataFrame for further analysis if needed

# --- Example Usage of the Evaluation Harness ---

# 1. Instantiate the Agent
mock_agent = MockAgent(name="BasicMockAgent", latency_range=(0.05, 0.2))

# 2. Instantiate the Evaluation Harness with test cases
harness = AgentEvaluationHarness(mock_test_cases)

# 3. Run the evaluation
harness.run_evaluation(mock_agent)

# 4. Generate and display the report
full_results_df = harness.generate_report()

# You can also inspect the raw results if needed
# print("\nRaw Results DataFrame Head:")
# print(full_results_df.head())
